**Get abstracts from CrossRef API**

In [ ]:
### API ###
# !pip install requests

import pandas as pd
import requests
import json
import re

def clean_abstract(text):
    return re.sub(r"<[^>]+>", "", text).strip()

response = requests.get("https://api.crossref.org/v1/works?filter=issn:0003-0554&sort=published&order=desc&rows=365&mailto=sam.kaenner@uni-konstanz.de").json()
articles = response["message"]["items"]

results = []

for a in articles:
    results.append({
        "title": a.get("title", [""])[0],
        "doi": a.get("DOI"),
        "published": a.get("published", {}).get("date-parts", [[]])[0],
        "abstract": clean_abstract(a.get("abstract", "No abstract available"))
    })

with open("data/apsr_abstracts.json", "w") as f:
    json.dump(results, f, indent=2)

**Load and clean abstracts**

In [46]:
with open("data/apsr_abstracts.json", "r") as f:
    results = json.load(f)

def parse_date(parts):
    if len(parts) >= 3:
        return pd.to_datetime(f"{parts[0]}-{parts[1]}-{parts[2]}")
    elif len(parts) == 2:
        return pd.to_datetime(f"{parts[0]}-{parts[1]}-01")
    else:
        return pd.to_datetime(f"{parts[0]}-01-01")

articles_df = pd.DataFrame(results)
articles_df["published"] = articles_df["published"].apply(parse_date)
articles_df.loc[articles_df["abstract"] == "No abstract available","abstract"] = pd.NA
articles_df = articles_df.dropna(subset=["abstract"])

**Generate worse abstracts using Ollama + Mistral 7b**

In [ ]:
import ollama

PROMPT = """You are rewriting a political science research abstract.
Rules:
- DO NOT under any circumstances change the findings or results of the original paper
- DO NOT overstate results, neither in scope nor size
- DO MAKE the abstract casually entertaining by making it intentionally terrible
- DO NOT use explicit words to state that your abstract is funny.
- SHORTEN the abstract
- DO NOT think of a title.
"""

model = "mistral:7b"

worse_abstracts = []

for abstract in articles_df["abstract"]:
    worse_abstract = ollama.chat(model=model, messages=[{"role": "user", "content": PROMPT.format(text=abstract)}], options={"temperature": 0}).message.content.strip()
    worse_abstracts.append(worse_abstract)


In [48]:
articles_df.assign(worse = worse_abstracts)

,title,doi,published,abstract,worse
0,Elite Partisan Disagreement and Military Victo...,10.1017/s0003055426101543,2026-03-24,Does partisan disagreement impact expectations...,In an audacious attempt to decipher the enigma...
1,The Effects of Exposure to New Electoral Rules...,10.1017/s0003055426101518,2026-03-23,How do electoral rules influence voters? We bu...,In an audacious attempt to decipher the enigma...
2,What Is the Wrong of Capitalism? A Reply to Ch...,10.1017/s0003055426101567,2026-03-23,Chiara Cordelli has recently criticized radica...,In an audacious attempt to decipher the enigma...
3,Do Elites Know Best? Candidate Selection and P...,10.1017/s0003055426101531,2026-03-23,How do candidate selection processes shape pol...,In an audacious attempt to decipher the enigma...
4,Exiting Russia,10.1017/s000305542610152x,2026-03-11,"As of February 24, 2022, over forty thousand f...",In an audacious attempt to decipher the enigma...
...,...,...,...,...,...
360,Female Representation and Legitimacy: Evidence...,10.1017/s0003055423000357,2023-04-27,How does the gender composition of deliberativ...,In an audacious attempt to decipher the enigma...
361,"Extraction, Assimilation, and Accommodation: T...",10.1017/s0003055423000333,2023-04-26,Why do some Indigenous communities experience ...,In an audacious attempt to decipher the enigma...
362,Fathers’ Leave Reduces Sexist Attitudes,10.1017/s0003055423000369,2023-04-26,Research shows that sexist attitudes are deepl...,In an audacious attempt to decipher the enigma...
363,Contested Killings: The Mobilizing Effects of ...,10.1017/s0003055423000321,2023-04-26,"Recently, we have witnessed the politicizing e...",In an audacious attempt to decipher the enigma...
